In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
resume_df = pd.read_csv("cleaned_resume1.csv")
resume_df = resume_df.dropna(subset=['cleaned_resume'])

print("Resume dataset shape:", resume_df.shape)
resume_df.head()

Resume dataset shape: (2483, 2)


,Category,cleaned_resume
0,HR,administrator marketing associate administrato...
1,HR,specialist operation summary versatile medium ...
2,HR,director summary year experience recruiting pl...
3,HR,specialist summary dedicated driven dynamic ye...
4,HR,manager skill highlight skill department start...


In [5]:
job_df = pd.read_csv("dataset/job_title_des.csv")

print("Job dataset shape:", job_df.shape)
print("Columns:", job_df.columns)

job_df.head()

Job dataset shape: (2277, 3)
Columns: Index(['Unnamed: 0', 'Job Title', 'Job Description'], dtype='str')


,Unnamed: 0,Job Title,Job Description
0,0,Flutter Developer,We are looking for hire experts flutter develo...
1,1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ..."
3,3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outsi...
4,4,Full Stack Developer,job responsibility full stack engineer – react...


In [6]:
job_df = job_df[['Job Title', 'Job Description']].copy()

print(job_df.shape)
job_df.head()

(2277, 2)


,Job Title,Job Description
0,Flutter Developer,We are looking for hire experts flutter develo...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ..."
3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outsi...
4,Full Stack Developer,job responsibility full stack engineer – react...


In [7]:
print(job_df.isnull().sum())


Job Title          0
Job Description    0
dtype: int64


In [8]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    words = [
        lemmatizer.lemmatize(word)
        for word in text.split()
        if word not in stop_words and len(word) > 2
    ]
    
    return " ".join(words)

In [9]:
job_df['cleaned_job'] = job_df['Job Description'].apply(clean_text)

job_df.head()

,Job Title,Job Description,cleaned_job
0,Flutter Developer,We are looking for hire experts flutter develo...,looking hire expert flutter developer eligible...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...,python django developer lead job code pdj stro...
2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ...",data scientist contractor bangalore responsibi...
3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outsi...,job description strong framework outside io al...
4,Full Stack Developer,job responsibility full stack engineer – react...,job responsibility full stack engineer react r...


In [10]:
all_text = pd.concat([
    resume_df['cleaned_resume'],
    job_df['cleaned_job']
])

tfidf = TfidfVectorizer(max_features=8000)

tfidf.fit(all_text)

print("Vocabulary size:", len(tfidf.get_feature_names_out()))

Vocabulary size: 8000


In [12]:
resume_vectors = tfidf.transform(resume_df['cleaned_resume'])
job_vectors = tfidf.transform(job_df['cleaned_job'])

print("Resume vectors shape:", resume_vectors.shape)
print("Job vectors shape:", job_vectors.shape)

Resume vectors shape: (2483, 8000)
Job vectors shape: (2277, 8000)


In [13]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(resume_vectors, job_vectors)

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (2483, 2277)


In [14]:
resume_index = 0

similar_scores = similarity_matrix[resume_index]

top_indices = similar_scores.argsort()[-5:][::-1]

recommended_jobs = job_df.iloc[top_indices][['Job Title']]

recommended_jobs

,Job Title
515,Java Developer
2141,Database Administrator
1880,Flutter Developer
1559,Database Administrator
980,Database Administrator


In [15]:
job_df.head()

,Job Title,Job Description,cleaned_job
0,Flutter Developer,We are looking for hire experts flutter develo...,looking hire expert flutter developer eligible...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...,python django developer lead job code pdj stro...
2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ...",data scientist contractor bangalore responsibi...
3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outsi...,job description strong framework outside io al...
4,Full Stack Developer,job responsibility full stack engineer – react...,job responsibility full stack engineer react r...


In [21]:
import joblib

model_xgb = joblib.load("xgb_model.pkl")
tfidf_cls = joblib.load("tfidf_classifier.pkl")
le = joblib.load("label_encoder.pkl")

print("Model loaded successfully.")

Model loaded successfully.


In [34]:
resume_df = pd.read_csv("cleaned_resume1.csv")
resume_df = resume_df.dropna(subset=['cleaned_resume'])

job_df = pd.read_csv("dataset/job_title_des.csv")
job_df = job_df[['Job Title', 'Job Description']].copy()

In [35]:
job_df['cleaned_job'] = job_df['Job Description'].apply(clean_text)

In [48]:
tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,2),
    min_df=3
)

all_text = pd.concat([
    resume_df['cleaned_resume'],
    job_df['cleaned_job']
])

tfidf.fit(all_text)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1

In [49]:
resume_vectors = tfidf.transform(resume_df['cleaned_resume'])
job_vectors = tfidf.transform(job_df['cleaned_job'])

In [50]:
from sklearn.metrics.pairwise import cosine_similarity

resume_index = 0  # test any resume

resume_vector = resume_vectors[resume_index]

similarity_scores = cosine_similarity(resume_vector, job_vectors)[0]

top_indices = similarity_scores.argsort()[-5:][::-1]

recommended_jobs = job_df.iloc[top_indices][['Job Title']]

recommended_jobs

,Job Title
2141,Database Administrator
980,Database Administrator
1454,Database Administrator
515,Java Developer
1559,Database Administrator


In [39]:
print(resume_df['Category'].iloc[0])

HR


In [40]:
print(resume_df['cleaned_resume'].iloc[0][:200])

administrator marketing associate administrator summary dedicated customer service manager year experience hospitality customer service management respected builder leader customer focused team strive


In [44]:
resume_df[resume_df['Category'] == "INFORMATION-TECHNOLOGY"].index[0]

np.int64(217)

In [51]:
resume_index = 217  # IT resume

resume_text = resume_df['cleaned_resume'].iloc[resume_index]

resume_vector = tfidf.transform([resume_text])

similarity_scores = cosine_similarity(resume_vector, job_vectors)[0]

top_indices = similarity_scores.argsort()[-5:][::-1]

recommended_jobs = job_df.iloc[top_indices][['Job Title']]

recommended_jobs

,Job Title
1459,Database Administrator
866,Software Engineer
1504,Database Administrator
1271,Database Administrator
580,Network Administrator


In [52]:
def recommend_by_index(resume_index, top_n=5):
    resume_text = resume_df['cleaned_resume'].iloc[resume_index]
    
    resume_vector = tfidf.transform([resume_text])
    similarity_scores = cosine_similarity(resume_vector, job_vectors)[0]
    
    top_indices = similarity_scores.argsort()[-top_n:][::-1]
    
    print("Actual Resume Category:", resume_df['Category'].iloc[resume_index])
    print("Top Recommended Jobs:")
    print(job_df.iloc[top_indices][['Job Title']])

In [53]:
recommend_by_index(0)      # HR
recommend_by_index(217)    # IT
recommend_by_index(500)    # Try random
recommend_by_index(1000)   # Another random

Actual Resume Category: HR
Top Recommended Jobs:
                   Job Title
2141  Database Administrator
980   Database Administrator
1454  Database Administrator
515           Java Developer
1559  Database Administrator
Actual Resume Category: INFORMATION-TECHNOLOGY
Top Recommended Jobs:
                   Job Title
1459  Database Administrator
866        Software Engineer
1504  Database Administrator
1271  Database Administrator
580    Network Administrator
Actual Resume Category: ADVOCATE
Top Recommended Jobs:
                   Job Title
1240    Full Stack Developer
980   Database Administrator
1289    Full Stack Developer
1888  Database Administrator
2089  Database Administrator
Actual Resume Category: SALES
Top Recommended Jobs:
                   Job Title
1315  Database Administrator
23    Database Administrator
1846  Database Administrator
1991       Software Engineer
930     Full Stack Developer


In [54]:
test_resume = """
Software engineer with 4 years experience in Python, Java,
SQL databases, backend development, cloud computing, AWS,
REST APIs and system architecture.
"""

resume_vector = tfidf.transform([test_resume])
similarity_scores = cosine_similarity(resume_vector, job_vectors)[0]

top_indices = similarity_scores.argsort()[-5:][::-1]

job_df.iloc[top_indices][['Job Title']]

,Job Title
1630,Backend Developer
1810,Node js developer
1565,Machine Learning
958,DevOps Engineer
850,DevOps Engineer


In [56]:
test_resume = """
Human resource manager with experience in recruitment,
employee onboarding, payroll management, HR compliance,
training and performance evaluation.
"""
resume_vector = tfidf.transform([test_resume])
similarity_scores = cosine_similarity(resume_vector, job_vectors)[0]

top_indices = similarity_scores.argsort()[-5:][::-1]

job_df.iloc[top_indices][['Job Title']]

,Job Title
859,Django Developer
1493,Database Administrator
1454,Database Administrator
1305,Database Administrator
101,Wordpress Developer


In [57]:
test_resume = """
Chartered accountant with expertise in auditing,
financial reporting, taxation, budgeting, compliance
and corporate finance.
"""
resume_vector = tfidf.transform([test_resume])
similarity_scores = cosine_similarity(resume_vector, job_vectors)[0]

top_indices = similarity_scores.argsort()[-5:][::-1]

job_df.iloc[top_indices][['Job Title']]

,Job Title
101,Wordpress Developer
13,Software Engineer
2149,Wordpress Developer
596,Software Engineer
1598,Wordpress Developer


In [58]:
for idx in top_indices:
    print(job_df.iloc[idx]['Job Title'],
          "Score:", round(similarity_scores[idx], 3))

Wordpress Developer Score: 0.088
Software Engineer Score: 0.062
Wordpress Developer Score: 0.061
Software Engineer Score: 0.05
Wordpress Developer Score: 0.049
